# 🏦 Training Hybrid IndoBERT + BERTopic
Notebook ini digunakan untuk melatih model pemodelan topik pada ulasan bank KBMI 4 menggunakan pendekatan Hybrid.

In [1]:
import pandas as pd
import yaml
import json
import os
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer

def load_config(config_path="config.yaml"):
    with open(config_path, "r") as f:
        return yaml.safe_load(f)

config = load_config()
print("Config loaded successfully.")

Config loaded successfully.


## 1. Load Data Bersih

In [2]:
data_path = config['data']['processed_file']
if not os.path.exists(data_path):
    print(f"File {data_path} tidak ditemukan. Pastikan sudah menjalankan preprocessing.ipynb")
else:
    df = pd.read_csv(data_path)
    docs = df['content'].astype(str).tolist()
    print(f"Berhasil memuat {len(docs)} ulasan.")
    display(df.head())

File data/processed/cleaned_reviews.csv tidak ditemukan. Pastikan sudah menjalankan preprocessing.ipynb


## 2. Inisialisasi Model Hybrid

In [3]:
# Embedding Model (IndoBERT)
print(f"Loading {config['indobert']['model_name']}...")
embedding_model = SentenceTransformer(config['indobert']['model_name'])

# UMAP (Dimensionality Reduction)
umap_model = UMAP(
    n_neighbors=config['bertopic']['umap']['n_neighbors'],
    n_components=config['bertopic']['umap']['n_components'],
    min_dist=config['bertopic']['umap']['min_dist'],
    metric=config['bertopic']['umap']['metric'],
    random_state=config['global']['seed']
)

# HDBSCAN (Clustering)
hdbscan_model = HDBSCAN(
    min_cluster_size=config['bertopic']['hdbscan']['min_cluster_size'],
    min_samples=config['bertopic']['hdbscan']['min_samples'],
    cluster_selection_method=config['bertopic']['hdbscan']['cluster_selection_method'],
    prediction_data=True
)

# Vectorizer
stopwords = []
if os.path.exists(config['preprocessing']['stopwords_file']):
    with open(config['preprocessing']['stopwords_file'], 'r') as f:
        stopwords = [line.strip() for line in f.readlines()]

vectorizer_model = CountVectorizer(stop_words=stopwords)

Loading indobenchmark/indobert-base-p1...


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/498M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/498M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

## 3. Training BERTopic

In [4]:
topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    nr_topics=config['bertopic']['nr_topics'],
    calculate_probabilities=config['bertopic']['calculate_probabilities'],
    verbose=True
)

topics, probs = topic_model.fit_transform(docs)
print("Training Selesai.")

NameError: name 'docs' is not defined

## 4. Visualisasi & Analisis

In [ ]:
# Tampilkan 10 topik teratas
info = topic_model.get_topic_info()
display(info.head(10))

# Visualisasi Interaktif
topic_model.visualize_topics()

In [ ]:
# Visualisasi Kata Kunci per Topik
topic_model.visualize_barchart(top_n_topics=10)

## 5. Simpan Model

In [ ]:
output_dir = config['global']['output_path']
os.makedirs(output_dir, exist_ok=True)

info.to_csv(os.path.join(output_dir, "topic_info.csv"), index=False)
topic_model.save(os.path.join(output_dir, "hybrid_bertopic_model"))
print(f"Model dan info topik disimpan di {output_dir}")